# 05 — Fusion Experiments

Experiment with multimodal fusion strategies:
- Concatenation vs Cross-Attention vs Gated fusion
- Ablation studies (text-only vs face-only vs fused)
- Hyperparameter tuning for fusion layer

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from ml.fusion.feature_fusion import ConcatFusion, CrossAttentionFusion, GatedFusion

# Compare fusion strategies
batch_size = 4
embed_dim = 256

text_emb = torch.randn(batch_size, embed_dim)
vision_emb = torch.randn(batch_size, embed_dim)

strategies = {
    'Concatenation': ConcatFusion(embed_dim),
    'Cross-Attention': CrossAttentionFusion(embed_dim),
    'Gated': GatedFusion(embed_dim),
}

for name, fusion in strategies.items():
    out = fusion(text_emb, vision_emb)
    params = sum(p.numel() for p in fusion.parameters())
    print(f'{name}: output={out.shape}, params={params:,}')

In [ ]:
# Full model forward pass
from ml.fusion.train_fusion_model import MultimodalModel

model = MultimodalModel()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters: {total_params - trainable_params:,}')

In [ ]:
# Test forward pass
from ml.text_pipeline.tokenizer import get_tokenizer

tokenizer = get_tokenizer('distilbert')
text = "I feel hopeless and tired"
enc = tokenizer(text, return_tensors='pt', max_length=128,
                padding='max_length', truncation=True)

dummy_img = torch.randn(1, 3, 224, 224)

model.eval()
with torch.no_grad():
    logits = model(enc['input_ids'], enc['attention_mask'], dummy_img)

probs = torch.softmax(logits, dim=1)
print(f'Output logits: {logits}')
print(f'Probabilities: {probs}')
print(f'Prediction: {"Depressed" if probs[0, 1] > 0.5 else "Non-Depressed"}')

## Next Steps

- Run full multimodal training with `src/fusion/train_fusion_model.py`
- Compare fusion strategies on real data
- Ablation: text-only vs face-only vs fused